In [1]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

from vgg_cifar import vgg13

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Load VGG13
model = vgg13()
model.load_state_dict(
    torch.load(
        "./model/cifar10_vgg13_acc_94.730.pt",
        map_location=device
    )
)
model = model.to(device)
model.eval()


/tmp/ipykernel_3527927/2904084351.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilati

In [4]:
# Same TEST transform used in main.py
test_dataset = datasets.CIFAR10(
    "./data.cifar10",
    train=False,
    download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),
            (0.2023, 0.1994, 0.2010)
        )
    ])
)

In [5]:
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

In [6]:
# Get first image
data, target = next(iter(test_loader))

x = data[0:1].to(device)
y = target[0:1].to(device)

with torch.no_grad():
    output = model(x)

print("True class:", y.item())

True class: 3


In [7]:
print("\nRaw VGG output (logits):")
print(output[0])

print("\nProbabilities:")
print(F.softmax(output, dim=1)[0])

print("\nLog probabilities:")
print(F.log_softmax(output, dim=1)[0])

print("\nPredicted class:")
print(output.argmax(dim=1).item())

print("\nOld NLL directly on logits:")
print(F.nll_loss(output, y).item())

print("\nCross Entropy:")
print(F.cross_entropy(output, y).item())

print("\nNLL after log_softmax:")
print(F.nll_loss(F.log_softmax(output, dim=1), y).item())


Raw VGG output (logits):
tensor([ 0.1431, -0.5849, -0.2003,  3.4463, -0.6168, -0.2217, -0.5564, -0.7841,
        -0.0420, -0.5815], device='cuda:0')

Probabilities:
tensor([0.0305, 0.0147, 0.0217, 0.8302, 0.0143, 0.0212, 0.0152, 0.0121, 0.0254,
        0.0148], device='cuda:0')

Log probabilities:
tensor([-3.4892, -4.2173, -3.8326, -0.1861, -4.2491, -3.8541, -4.1887, -4.4164,
        -3.6744, -4.2138], device='cuda:0')

Predicted class:
3

Old NLL directly on logits:
-3.4462831020355225

Cross Entropy:
0.1860675811767578

NLL after log_softmax:
0.1860675811767578
